<a href="https://colab.research.google.com/github/golammoula287/Thesis/blob/Assignment_Practice/transfer_learning_with_tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow-hub

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras import layers
import zipfile
import datetime
import os
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
print("TensorFlow version:", tf.__version__)

# Step 2: Download & extract data (skip if already exists)
if not os.path.exists("10_food_classes_10_percent"):
    !wget -q https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip
    with zipfile.ZipFile("10_food_classes_10_percent.zip", 'r') as zip_ref:
        zip_ref.extractall()
    print("Data extracted!")
else:
    print("Dataset already exists.")


TensorFlow version: 2.19.0
Dataset already exists.


In [ ]:
# Step 3: Paths
train_dir = "10_food_classes_10_percent/train"
test_dir = "10_food_classes_10_percent/test"


In [ ]:

# Step 4: Data generators
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

Found 750 images belonging to 10 classes.
Found 2500 images belonging to 10 classes.


In [ ]:
# Step 5: TensorBoard callback
def create_tensorboard_callback(name):
    log_dir = "logs/" + name + "/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    return tf.keras.callbacks.TensorBoard(log_dir=log_dir)

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras import layers

# Step 6: Model function
def create_model(model_url, num_classes):
    # Use the Keras Functional API for better compatibility
    inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))
    feature_extractor_layer = hub.KerasLayer(
        model_url,
        trainable=False, # Freeze the pre-trained weights
        name='feature_extraction_layer'
    )(inputs) # Removed explicit training=False from the call

    outputs = layers.Dense(num_classes, activation="softmax")(feature_extractor_layer)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    model.compile(
        loss="categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"]
    )

    return model

In [ ]:
# Step 7: Model URLs
resnet_url = "https://tfhub.dev/google/imagenet/resnet_v2_50/feature_vector/4"
efficientnet_url = "https://tfhub.dev/tensorflow/efficientnet/b0/feature-vector/1"


In [ ]:
# Step 8: Create models
resnet_model = create_model(resnet_url, train_data.num_classes)
efficientnet_model = create_model(efficientnet_url, train_data.num_classes)

ValueError: Exception encountered when calling layer 'feature_extraction_layer' (type KerasLayer).

A KerasTensor is symbolic: it's a placeholder for a shape an a dtype. It doesn't have any actual numerical value. You cannot convert it to a NumPy array.

Call arguments received by layer 'feature_extraction_layer' (type KerasLayer):
  • inputs=<KerasTensor shape=(None, 224, 224, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor_11>
  • training=None